In [82]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# Carregando Dataset
df_sonar = pd.read_csv('sonar_dataset.csv', sep=',',header=None)
df_sonar
X, y = df_sonar.iloc[:, 0:59], df_sonar[60].apply(lambda x: 1 if x == 'R' else 0)

# Pipeline - Pré-processamento
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clf', DecisionTreeClassifier(random_state=7))
])

# Parametros para o Classificador
kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

parametros = {
    'pca__n_components': [3, 5, 7, 10, 20],
    'clf__max_depth': [3, 5, 7, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__criterion': ['gini', 'entropy'],
    'clf__ccp_alpha': [0.0, 0.001, 0.01, 0.1]  # Pruning (poda na arvore de decisão)
}

scores = [
    'accuracy',
    'precision',
    'recall',
    'balanced_accuracy', # Média entre Especififdade e sensitividade
    'f1',
    'roc_auc'
]
grid_search = GridSearchCV(
    pipeline,
    param_grid=parametros,
    scoring=scores,
    cv=kfolds,
    refit=False
)
grid_search.fit(X,y)

resultados = pd.DataFrame(grid_search.cv_results_)

# 25 melhores modelos pelo indice f1 para conferir figuras de mérito de cada métrica

# resultados = resultados.sort_values(by='rank_test_f1')
# resultados.head(25)[[
#     'rank_test_accuracy',
#     'rank_test_precision',
#     'rank_test_recall',
#     'rank_test_balanced_accuracy',
#     'rank_test_f1',
#     'rank_test_roc_auc'
# ]]

best_results = {}
best_idx = resultados['rank_test_f1'].argmin() 
best_results['params'] = resultados['params'][best_idx]
for metrica in scores:
    
    best_results[metrica] = {
        'score': resultados.loc[best_idx, f'mean_test_{metrica}'],
        'std': resultados.loc[best_idx, f'std_test_{metrica}']
    }

    
print(f"PARAMS: {best_results['params']}")
for metric, data in best_results.items():
    if(metric != 'params'):
        print(f"\n{metric.upper()}:")
        print(f"Score: {data['score']:.4f} ± {data['std']:.4f}")
    

PARAMS: {'clf__ccp_alpha': 0.01, 'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 5, 'pca__n_components': 5}

ACCURACY:
Score: 0.7979 ± 0.0778

PRECISION:
Score: 0.7752 ± 0.0916

RECALL:
Score: 0.8032 ± 0.0791

BALANCED_ACCURACY:
Score: 0.7980 ± 0.0775

F1:
Score: 0.7877 ± 0.0810

ROC_AUC:
Score: 0.8383 ± 0.0813
